# What is a Neuron?

This notebook accompanies the **ML Viz** lesson on artificial neurons.
We'll implement a single neuron from scratch and visualize how it works.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/neural-networks/01-what-is-a-neuron

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — the atom of a neural network

A **neuron** is the simplest learnable unit: it takes a weighted sum of its inputs, adds a
bias, and squashes the result through a non-linear **activation**. That's it — `y = f(w·x + b)`.
On its own a neuron can only draw a straight line (it's literally logistic regression). The magic
is **composition**: stack a layer of neurons with a non-linear activation between them and the
network can carve *curved* decision boundaries that no single neuron could. This notebook builds a
neuron from scratch, proves why the non-linearity is essential, then assembles a 2-layer network —
deriving backprop by hand — and checks it against `sklearn`.

## The neuron equation

$$y = f\left(\sum_{i=1}^n w_i x_i + b\right)$$

Three parts:
- **Inputs** $x_i$ — the data coming in
- **Weights** $w_i$ — how much each input matters
- **Bias** $b$ — shifts the activation threshold
- **Activation** $f$ — introduces non-linearity

In [ ]:
class Neuron:
    def __init__(self, weights, bias, activation='relu'):
        self.weights = np.array(weights, dtype=float)
        self.bias = float(bias)
        self.activation = activation

    def _activate(self, z):
        if self.activation == 'relu':
            return np.maximum(0, z)
        elif self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-z))
        elif self.activation == 'tanh':
            return np.tanh(z)
        return z  # linear

    def forward(self, x):
        z = np.dot(self.weights, x) + self.bias
        return self._activate(z), z  # return (output, pre-activation)


# A fully worked forward pass (matches the lesson):
#   weights = [2, -1, 0.5], bias = 1, inputs = [3, 2, 4]
w = [2, -1, 0.5]
b = 1
x = [3, 2, 4]

# Step 1 - dot product, term by term
terms = [wi * xi for wi, xi in zip(w, x)]
print('Term-by-term:  {} = {}'.format(' + '.join('({})({})'.format(wi, xi)
                                                  for wi, xi in zip(w, x)),
                                      sum(terms)))   # (2)(3) + (-1)(2) + (0.5)(4) = 6

# Step 2 + 3 - bias, then activation, for each activation type
for act in ('relu', 'sigmoid', 'tanh'):
    neuron = Neuron(weights=w, bias=b, activation=act)
    output, z = neuron.forward(x)
    print('{:>8}:  z = {}   ->   y = {:.4f}'.format(act, z, float(output)))
# Expected:  z = 7.0  ->  relu 7.0, sigmoid ~0.9991, tanh ~1.0000

## How changing one weight changes the output

Keep the inputs and bias fixed, and nudge only the first weight from $2$ to $3$.
Because $z$ depends on $w_1$ only through the term $w_1 x_1$, increasing $w_1$ by $1$
changes $z$ by exactly $x_1$. In other words $\partial z / \partial w_i = x_i$ — the
weight on a large input has a large influence on the output.

In [ ]:
z_before = np.dot([2, -1, 0.5], [3, 2, 4]) + 1   # original w1 = 2
z_after  = np.dot([3, -1, 0.5], [3, 2, 4]) + 1   # w1 bumped to 3

print('z before (w1=2) = {}'.format(z_before))    # 7.0
print('z after  (w1=3) = {}'.format(z_after))     # 10.0
print('change in z     = {}  (equals x[0] = {})'.format(z_after - z_before, 3))  # 3.0

## Activation functions compared

Let's plot all four common activation functions side by side.

In [ ]:
x = np.linspace(-5, 5, 300)

activations = {
    'ReLU':    np.maximum(0, x),
    'Sigmoid': 1 / (1 + np.exp(-x)),
    'Tanh':    np.tanh(x),
    'Linear':  x,
}
colors = ['#818cf8', '#14b8a6', '#eab308', '#f97316']

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
fig.suptitle('Activation Functions', color='white', fontsize=13, y=1.02)

for ax, (name, y), color in zip(axes, activations.items(), colors):
    ax.plot(x, y, color=color, linewidth=2.5)
    ax.axhline(0, color='#2e3347', linewidth=0.8)
    ax.axvline(0, color='#2e3347', linewidth=0.8)
    ax.set_title(name, color='white', fontsize=11)
    ax.set_xlim(-5, 5)

plt.tight_layout()
plt.show()

**What to notice:** the four activations behave very differently. **ReLU** is a simple hinge
(0 for negatives, identity for positives) — cheap and the modern default. **Sigmoid** and **tanh**
are smooth but **saturate** in the tails (flat → near-zero gradient → the vanishing-gradient
problem). **Linear** does nothing non-linear at all — which the next section shows is fatal.

## Why non-linearity matters

Without activation functions, stacking neurons is equivalent to a single linear transformation.
Let's prove this numerically.

In [ ]:
# Two-layer network without activation: W2 @ (W1 @ x + b1) + b2 = (W2@W1)@x + const
# This collapses to a single linear transformation.

np.random.seed(42)
W1 = np.random.randn(4, 2)   # 4 hidden neurons, 2 inputs
b1 = np.random.randn(4)
W2 = np.random.randn(1, 4)   # 1 output
b2 = np.random.randn(1)

x_test = np.array([3.0, -1.5])

# Without activation (linear)
h_linear = W1 @ x_test + b1          # hidden layer, no activation
y_linear = W2 @ h_linear + b2

# Equivalent single-layer
W_collapsed = W2 @ W1
b_collapsed = W2 @ b1 + b2
y_single = W_collapsed @ x_test + b_collapsed

print(f'Two-layer (no activation): {y_linear[0]:.6f}')
print(f'Collapsed single layer:    {y_single[0]:.6f}')
print(f'Difference (should be ~0): {abs(y_linear[0] - y_single[0]):.2e}')

# With ReLU — can no longer collapse
h_relu = np.maximum(0, W1 @ x_test + b1)
y_relu = W2 @ h_relu + b2
print(f'\nTwo-layer (with ReLU):     {y_relu[0]:.6f}  ← different!')

## What a single neuron can learn

A single neuron with sigmoid activation is equivalent to **logistic regression** — a linear decision boundary.

In [ ]:
# Generate linearly separable data
np.random.seed(7)
n = 80
X_pos = np.random.randn(n // 2, 2) + [1.5, 1.5]
X_neg = np.random.randn(n // 2, 2) + [-1.5, -1.5]
X = np.vstack([X_pos, X_neg])
y = np.array([1] * (n // 2) + [0] * (n // 2))

# Train a single sigmoid neuron via gradient descent
w = np.zeros(2)
b = 0.0
lr = 0.1

for _ in range(200):
    z = X @ w + b
    pred = 1 / (1 + np.exp(-z))
    err = pred - y
    w -= lr * X.T @ err / n
    b -= lr * err.mean()

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(*X_pos.T, color='#818cf8', s=30, label='Class 1', alpha=0.8)
ax.scatter(*X_neg.T, color='#f43f5e', s=30, label='Class 0', alpha=0.8)

# Decision boundary: w[0]*x + w[1]*y + b = 0  →  y = -(w[0]*x + b) / w[1]
xline = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 100)
yline = -(w[0] * xline + b) / w[1]
ax.plot(xline, yline, '--', color='#14b8a6', linewidth=2, label='Decision boundary')
ax.legend()
ax.set_title('Single Neuron: Linear Decision Boundary', color='white')
plt.tight_layout()
plt.show()
print(f'Learned weights: {w.round(3)}, bias: {b:.3f}')

**What to notice:** a single sigmoid neuron learns a **straight-line** decision boundary —
it *is* logistic regression. That's perfect for this linearly-separable data, but it's the ceiling
for one neuron: no straight line can separate data that's arranged in rings, which is exactly the
next experiment.

## A full neural network from scratch (2-layer MLP)

A single neuron only draws a straight line. Stack a hidden layer of neurons with a non-linear activation and you can carve **curved** boundaries. Here we build a 2-layer MLP, derive **backprop** by hand, and train it on a non-linearly-separable dataset.

In [ ]:
# Non-linearly-separable data: two concentric rings (inner=0, outer=1)
def make_rings(n=400, seed=0):
    rng = np.random.RandomState(seed)
    r = np.r_[rng.uniform(0, 1.0, n//2), rng.uniform(1.8, 2.8, n//2)]
    th = rng.uniform(0, 2*np.pi, n)
    X = np.c_[r*np.cos(th), r*np.sin(th)]
    y = np.r_[np.zeros(n//2), np.ones(n//2)].reshape(-1, 1)
    return X, y

X, y = make_rings()
plt.scatter(*X[y.ravel()==0].T, s=12, color='#818cf8', label='class 0')
plt.scatter(*X[y.ravel()==1].T, s=12, color='#f43f5e', label='class 1')
plt.legend(); plt.title('Not linearly separable — a single neuron cannot solve this'); plt.show()

### Forward pass, loss, and backprop

Two layers: $h=\tanh(XW_1+b_1)$, $\hat y=\sigma(hW_2+b_2)$, binary cross-entropy loss. Backprop is the chain rule: the output error $\hat y-y$ flows back through $W_2$, gets multiplied by the $\tanh'$ of the hidden layer, and updates $W_1$.

In [ ]:
def sigmoid(z): return 1/(1+np.exp(-z))

class MLP:
    def __init__(self, n_in=2, n_hidden=16, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, n_hidden) * np.sqrt(2/n_in)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.randn(n_hidden, 1) * np.sqrt(2/n_hidden)
        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        self.X = X
        self.z1 = X @ self.W1 + self.b1
        self.h  = np.tanh(self.z1)
        self.p  = sigmoid(self.h @ self.W2 + self.b2)
        return self.p

    def backward(self, y, lr):
        n = y.shape[0]
        dz2 = (self.p - y) / n                 # dL/d(output pre-activation)
        dW2 = self.h.T @ dz2; db2 = dz2.sum(0, keepdims=True)
        dh  = dz2 @ self.W2.T
        dz1 = dh * (1 - self.h**2)             # times tanh'(z1)
        dW1 = self.X.T @ dz1; db1 = dz1.sum(0, keepdims=True)
        self.W2 -= lr*dW2; self.b2 -= lr*db2
        self.W1 -= lr*dW1; self.b1 -= lr*db1

def bce(p, y):
    return -np.mean(y*np.log(p+1e-9) + (1-y)*np.log(1-p+1e-9))

net = MLP(n_hidden=16)
losses = []
for epoch in range(2000):
    p = net.forward(X)
    losses.append(bce(p, y))
    net.backward(y, lr=0.5)
acc = ((net.forward(X) > 0.5) == y).mean()
print(f'final loss {losses[-1]:.4f}   training accuracy {acc:.3f}')

### The learned non-linear boundary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(losses, color='#14b8a6'); ax[0].set_title('Training loss')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('BCE')

xx, yy = np.meshgrid(np.linspace(-3.2, 3.2, 250), np.linspace(-3.2, 3.2, 250))
Z = net.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax[1].contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.7)
ax[1].scatter(*X[y.ravel()==0].T, s=10, color='#818cf8')
ax[1].scatter(*X[y.ravel()==1].T, s=10, color='#f43f5e')
ax[1].set_title('MLP decision boundary (a single neuron could never do this)')
plt.tight_layout(); plt.show()

**What to notice:** the 2-layer MLP carved a **curved, closed** boundary that wraps the inner
ring — something no single neuron (straight line) could do. The training-loss curve falls as
backprop pushes the `ŷ − y` error signal back through `W2` and `W1`. Depth + non-linearity is
what turned a line into a ring.

## The library way — validate against `sklearn`

Two checks close the loop. First, a **single linear neuron** (logistic regression) *cannot* solve
the rings — `sklearn.LogisticRegression` scores near chance. Second, our from-scratch MLP should
match `sklearn`'s `MLPClassifier` on the same data. Both confirm the "depth beats a line" story.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

yr = y.ravel()

# 1) one neuron == logistic regression -> fails on the rings (not linearly separable)
lin_acc = LogisticRegression().fit(X, yr).score(X, yr)
print(f'single linear neuron (LogisticRegression) accuracy on rings: {lin_acc:.2f}  (~chance)')

# 2) our from-scratch MLP vs sklearn's MLPClassifier
our_acc = ((net.forward(X).ravel() > 0.5).astype(int) == yr).mean()
sk_acc = MLPClassifier(hidden_layer_sizes=(16,), activation='tanh',
                       max_iter=3000, random_state=0).fit(X, yr).score(X, yr)
print(f'our from-scratch MLP accuracy:  {our_acc:.2f}')
print(f'sklearn MLPClassifier accuracy: {sk_acc:.2f}')
assert lin_acc < 0.7 and our_acc > 0.9 and sk_acc > 0.9, "line fails, both MLPs succeed"
print('\na single neuron can not separate rings; a 2-layer MLP can — matching sklearn ✓')

**What to notice:** the linear neuron scores ~0.5 (coin flip) on the rings, while both our MLP
and sklearn's clear 0.9+. Same data, same task — the only difference is the hidden layer and its
non-linearity. That gap *is* the reason neural networks exist.

## Gotchas & tradeoffs

- **No non-linearity = no depth.** Without an activation, stacked layers collapse to a single
  linear map (`W₂W₁x + c`) — proven above. The activation is what makes depth meaningful.
- **Saturation kills gradients.** Sigmoid/tanh flatten in the tails, so deep stacks of them
  suffer vanishing gradients — a big reason ReLU took over.
- **Dead ReLUs.** A neuron stuck with negative pre-activations for every input outputs 0 forever
  and never learns (mitigated by careful init, LeakyReLU).
- **Scale matters.** Un-normalized inputs or bad weight init push neurons into saturation from the
  start — hence input normalization and He/Xavier initialization.

In [ ]:
# Without an activation, two layers ARE one layer (a single linear map)
rng = np.random.default_rng(0)
W1, b1 = rng.normal(size=(4, 2)), rng.normal(size=4)
W2, b2 = rng.normal(size=(1, 4)), rng.normal(size=1)
x = rng.normal(size=2)

two_layer_linear = W2 @ (W1 @ x + b1) + b2          # no activation
collapsed        = (W2 @ W1) @ x + (W2 @ b1 + b2)   # one equivalent linear map
print('two linear layers == one linear map?', np.allclose(two_layer_linear, collapsed))

**What to notice:** with no activation, the two-layer computation equals a single linear map
`(W₂W₁)x + const` — exactly. Adding more linear layers buys you *nothing*; only the non-linear
activation lets extra layers add real modeling power.

## Key takeaways

- A neuron computes $y = f(\sum_i w_i x_i + b)$ — a weighted sum passed through an activation.
- **Weights** scale each input; the **bias** shifts the threshold.
- The **activation** $f$ adds non-linearity; without it, stacked layers collapse to one linear map.
- A single sigmoid neuron *is* logistic regression — a linear decision boundary.
- Depth + non-linearity is what lets networks model complex, hierarchical functions.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The neuron equation

A neuron is a weighted sum squashed through an activation:

$$y = \sigma\!\left(\mathbf{w} \cdot \mathbf{x} + b\right), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

Implement it with a sigmoid. The checks pin the landmarks: $z = 0$ gives exactly $0.5$, and large $|z|$ saturates toward 0 or 1.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def neuron(x, w, b):
    """Sigmoid neuron: sigmoid(w . x + b)."""
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    # TODO(you): the pre-activation z = w . x + b (hint: np.dot)
    z = ...

    # TODO(you): squash it
    return ...

In [ ]:
# Checks — run me
assert abs(neuron([1, 1], [2, -2], 0.0) - 0.5) < 1e-12, "z = 0 -> sigmoid gives exactly 0.5"
assert neuron([3, 4], [10, 10], 0.0) > 0.999, "large positive z saturates toward 1"
assert neuron([3, 4], [-10, -10], 0.0) < 0.001, "large negative z saturates toward 0"

expected = sigmoid(0.5 * 2 + (-1) * 1 + 0.5)
assert abs(neuron([2, 1], [0.5, -1], 0.5) - expected) < 1e-12, "must match the neuron equation"

# Numerical-stability edge case: |z| large enough to fully saturate but must
# still come back as exactly 0/1, never NaN (a naive exp() can overflow here).
with np.errstate(over='ignore'):
    big   = neuron([100, 100], [5, 5], 0.0)     # z = 1000
    small = neuron([100, 100], [-5, -5], 0.0)   # z = -1000
assert big == 1.0 and small == 0.0, "extreme |z| must saturate to exactly 0/1, not NaN"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def neuron(x, w, b):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    z = np.dot(w, x) + b
    return sigmoid(z)
```

</details>

### Exercise 2 — A perceptron is a line

With a hard threshold instead of a sigmoid, the neuron becomes a **perceptron**: predict 1 when $\mathbf{w} \cdot \mathbf{x} + b > 0$. One weight vector = one line through input space — enough for AND, OR, and NOR, as the checks confirm by truth table. The last check restates the classic limit: no single line computes XOR (that's why we need hidden layers).

In [ ]:
def perceptron_predict(X, w, b):
    """Hard-threshold predictions (0/1) for a batch of inputs X."""
    X = np.asarray(X, dtype=float)
    w = np.asarray(w, dtype=float)

    # TODO(you): scores X @ w + b, then compare > 0 and cast with .astype(int)
    return ...

In [ ]:
# Checks — run me
gates_in = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

assert list(perceptron_predict(gates_in, [1, 1], -1.5)) == [0, 0, 0, 1], "w=[1,1], b=-1.5 is the AND gate"
assert list(perceptron_predict(gates_in, [1, 1], -0.5)) == [0, 1, 1, 1], "w=[1,1], b=-0.5 is the OR gate"
assert list(perceptron_predict(gates_in, [-1, -1], 0.5)) == [1, 0, 0, 0], "w=[-1,-1], b=0.5 is NOR"

xor = [0, 1, 1, 0]
assert list(perceptron_predict(gates_in, [1, 1], -0.5)) != xor and \
       list(perceptron_predict(gates_in, [1, -1], 0.0)) != xor, "no single line computes XOR"

# Boundary edge case: exactly on the decision boundary (w.x + b == 0) must
# predict 0 -- the rule is a strict ">", not ">=".
assert list(perceptron_predict(np.array([[0, 0]]), [1, 1], 0.0)) == [0], \
    "exactly on the boundary (z=0) predicts class 0, not 1"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def perceptron_predict(X, w, b):
    X = np.asarray(X, dtype=float)
    w = np.asarray(w, dtype=float)
    return (X @ w + b > 0).astype(int)
```

</details>

---
## 🔬 Extra practice — an activation-function bank (Open-Deep-ML)

[Open-Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) has a whole family
of one-function activation problems. Rather than a separate scaffold for each of
the twelve, implement them all in one bank and compare them on shared axes:

**Bounded / S-shaped:**
[`22` sigmoid](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/22_sigmoid-activation-function-understanding),
[`96` hard sigmoid](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/96_implement-the-hard-sigmoid-activation-function),
[`100` softsign](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/100_implement-the-softsign-activation-function)

**ReLU family (unbounded above, clamped or leaky below):**
[`42` ReLU](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/42_implement-relu-activation-function),
[`44` leaky ReLU](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/44_leaky-relu-activation-function),
[`97` ELU](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/97_implement-the-elu-activation-function),
[`98` PReLU](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/98_implement-the-prelu-activation-function),
[`99` softplus](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/99_implement-the-softplus-activation-function),
[`102` swish](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/102_implement-the-swish-activation-function),
[`103` SELU](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/103_implement-the-selu-activation-function)

**Whole-vector / whole-model (not a curve in `x`, tested numerically instead of plotted):**
[`23` softmax](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/23_softmax-activation-function-implementation),
[`24` single neuron model](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/24_single-neuron)

All twelve are implemented in the one cell below, plotted together, then checked
against DML's own published test cases (from `tests.json`) plus a few
large-magnitude checks for saturation and numerical stability.

In [ ]:
def sigmoid(z):
    """DML #22: sigmoid(z) = 1 / (1 + e^-z)."""
    z = np.asarray(z, dtype=float)
    with np.errstate(over='ignore'):
        # TODO(you): 1 / (1 + exp(-z))
        return ...


def softmax(scores):
    """DML #23: softmax turns scores into a probability distribution."""
    scores = np.asarray(scores, dtype=float)
    shifted = scores - np.max(scores)          # subtract max for numerical stability
    # TODO(you): exponentiate `shifted`, normalize by its sum, round each entry to 4 decimals
    exps = ...
    probs = ...
    return [round(float(p), 4) for p in probs]


def single_neuron_model(features, labels, weights, bias):
    """DML #24: one sigmoid neuron's predictions + MSE against true labels."""
    features = np.asarray(features, dtype=float)
    labels = np.asarray(labels, dtype=float)
    weights = np.asarray(weights, dtype=float)
    # TODO(you): pre-activation z = X @ w + b, squashed through sigmoid() above
    z = ...
    probabilities = ...
    mse = float(np.mean((probabilities - labels) ** 2))
    return [round(float(p), 4) for p in probabilities], round(mse, 4)


def relu(z):
    """DML #42."""
    z = np.asarray(z, dtype=float)
    # TODO(you): max(0, z), elementwise (hint: np.maximum)
    return ...


def leaky_relu(z, alpha=0.01):
    """DML #44: like ReLU, but negative inputs leak a small slope instead of dying."""
    z = np.asarray(z, dtype=float)
    # TODO(you): z where z > 0, else alpha * z (hint: np.where)
    return ...


def hard_sigmoid(x):
    """DML #96: a piecewise-linear stand-in for sigmoid, cheap to compute."""
    x = np.asarray(x, dtype=float)
    # TODO(you): 0.2*x + 0.5, clipped to [0, 1] (hint: np.clip)
    return ...


def elu(x, alpha=1.0):
    """DML #97: smooth like ReLU for x>0, saturates to -alpha as x -> -inf."""
    x = np.asarray(x, dtype=float)
    with np.errstate(over='ignore'):
        # TODO(you): x where x > 0, else alpha * (exp(x) - 1).
        # Clip the exponent to <= 0 with np.minimum(x, 0) first so a large
        # positive x in the unused branch can never overflow.
        return ...


def prelu(x, alpha=0.25):
    """DML #98: leaky ReLU with a *learnable* negative slope alpha."""
    x = np.asarray(x, dtype=float)
    # TODO(you): x where x >= 0, else alpha * x
    return ...


def softplus(x):
    """DML #99: a smooth approximation of ReLU, log(1 + e^x)."""
    x = np.asarray(x, dtype=float)
    with np.errstate(over='ignore'):
        # TODO(you): the numerically-stable form max(x, 0) + log1p(exp(-|x|))
        # (the naive log(1 + exp(x)) overflows for x as small as ~710)
        return ...


def softsign(x):
    """DML #100: like tanh, but with polynomial (not exponential) tails."""
    x = np.asarray(x, dtype=float)
    # TODO(you): x / (1 + |x|)
    return ...


def swish(x):
    """DML #102: x * sigmoid(x) -- a smooth, self-gated activation."""
    x = np.asarray(x, dtype=float)
    # TODO(you): x * sigmoid(x)
    return ...


def selu(x):
    """DML #103: a self-normalizing ELU variant with fixed scale/alpha constants."""
    x = np.asarray(x, dtype=float)
    alpha = 1.6732632423543772
    scale = 1.0507009873554804
    with np.errstate(over='ignore'):
        # TODO(you): scale * (x where x > 0, else alpha * (exp(x) - 1))
        # (reuse the same np.minimum(x, 0) overflow guard as elu() above)
        return ...

In [ ]:
x = np.linspace(-5, 5, 400)

curves = [
    ('Sigmoid #22',      sigmoid(x),                '#2a78d6', '-'),
    ('Hard Sigmoid #96', hard_sigmoid(x),            '#1baf7a', '-'),
    ('Softsign #100',    softsign(x),                '#eda100', '-'),
    ('ReLU #42',         relu(x),                    '#4a3aa7', '--'),
    ('Leaky ReLU #44',   leaky_relu(x, alpha=0.15),  '#e34948', '--'),
    ('ELU #97',          elu(x),                     '#e87ba4', '--'),
    ('PReLU #98',        prelu(x),                   '#eb6834', '--'),
    ('Softplus #99',     softplus(x),                '#008300', '--'),
    ('Swish #102',       swish(x),                   '#9085e9', ':'),
    ('SELU #103',        selu(x),                    '#d95926', ':'),
]

fig, ax = plt.subplots(figsize=(9, 6))
for name, y, color, style in curves:
    ax.plot(x, y, label=name, color=color, linestyle=style, linewidth=2.2)
ax.axhline(0, color='#2e3347', linewidth=0.8)
ax.axvline(0, color='#2e3347', linewidth=0.8)
ax.set_ylim(-3, 5)
ax.set_xlabel('x'); ax.set_ylabel('activation(x)')
ax.set_title('10 activation functions from Open-Deep-ML, overlaid', color='white')
ax.legend(loc='upper left', fontsize=8, ncol=2, framealpha=0.9)
plt.tight_layout()
plt.show()

# softmax (#23) and single_neuron_model (#24) aren't 1D curves in x -- they act
# on whole vectors -- so demonstrate them numerically instead of plotting:
print('softmax([1, 2, 3])       =', softmax([1, 2, 3]))
probs, mse = single_neuron_model(
    [[0.5, 1.0], [-1.5, -2.0], [2.0, 1.5]], [0, 1, 0], [0.7, -0.4], -0.1)
print('single_neuron_model(...) =', probs, ' mse =', mse)

In [ ]:
# Checks — run me (grounded in DML's own tests.json, plus large-magnitude
# stability checks for saturation/overflow behavior)

# #22 sigmoid
assert abs(sigmoid(0) - 0.5) < 1e-9
assert abs(sigmoid(1) - 0.7311) < 1e-4 and abs(sigmoid(-1) - 0.2689) < 1e-4
assert sigmoid(1000) > 0.9999999 and sigmoid(-1000) < 1e-9, "saturates without overflowing"

# #23 softmax
assert softmax([1, 2, 3]) == [0.09, 0.2447, 0.6652]
assert softmax([1, 1, 1]) == [0.3333, 0.3333, 0.3333]
assert abs(sum(softmax([-5, 0, 5, 10])) - 1.0) < 1e-9, "always sums to 1"

# #24 single neuron model
probs, mse = single_neuron_model([[0.5, 1.0], [-1.5, -2.0], [2.0, 1.5]], [0, 1, 0], [0.7, -0.4], -0.1)
assert probs == [0.4626, 0.4134, 0.6682] and mse == 0.3349

# #42 ReLU / #44 leaky ReLU
assert relu(0) == 0.0 and relu(1) == 1.0 and relu(-1) == 0.0
assert leaky_relu(5) == 5.0 and leaky_relu(-1) == -0.01 and leaky_relu(-2, alpha=0.1) == -0.2
assert relu(-1000) == 0.0 and leaky_relu(-1000, alpha=0.01) == -10.0, "no clamping surprises at scale"

# #96 hard sigmoid -- including both clip boundaries
assert abs(hard_sigmoid(0.56) - 0.612) < 1e-4
assert hard_sigmoid(3.0) == 1.0 and hard_sigmoid(-1.0) == 0.3
assert hard_sigmoid(2.5) == 1.0 and hard_sigmoid(-2.5) == 0.0, "exact clip boundaries"
assert hard_sigmoid(1e6) == 1.0 and hard_sigmoid(-1e6) == 0.0, "stays clipped at extreme x"

# #97 ELU / #98 PReLU
assert elu(0) == 0.0 and abs(elu(-1) - (-0.6321)) < 1e-4 and abs(elu(-1, alpha=2.0) - (-1.2642)) < 1e-4
assert abs(elu(-1e6) - (-1.0)) < 1e-6, "ELU saturates to -alpha as x -> -inf, never overflows"
assert prelu(2.0) == 2.0 and prelu(-2.0) == -0.5 and prelu(-2.0, alpha=1.0) == -2.0

# #99 softplus -- large-magnitude numerical stability is the whole point of this one
assert abs(softplus(0) - 0.6931) < 1e-4 and abs(softplus(2) - 2.1269) < 1e-4
assert abs(softplus(100) - 100.0) < 1e-6 and abs(softplus(-100) - 0.0) < 1e-6
assert abs(softplus(1e6) - 1e6) < 1e-6 and abs(softplus(-1e6) - 0.0) < 1e-9, \
    "a naive log(1+exp(x)) would overflow long before x=1e6"

# #100 softsign
assert softsign(0) == 0.0 and softsign(1) == 0.5
assert abs(softsign(100) - 0.9901) < 1e-4 and abs(softsign(1e6) - 1.0) < 1e-4

# #102 swish / #103 SELU
assert abs(swish(1) - 0.7311) < 1e-4 and abs(swish(-10) - (-0.0005)) < 1e-4
assert abs(swish(-1e6) - 0.0) < 1e-6, "swish saturates to 0 for very negative x"
assert abs(selu(1.0) - 1.0507) < 1e-4 and abs(selu(-1.0) - (-1.1113)) < 1e-4
assert abs(selu(-1e6) - (-1.7580993408473766)) < 1e-6, "SELU saturates to -scale*alpha, still finite"

print("✅ Activation-function bank passed (DML #22, #23, #24, #42, #44, #96–#100, #102, #103)")

<details>
<summary>💡 Show solution</summary>

```python
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    with np.errstate(over='ignore'):
        return 1.0 / (1.0 + np.exp(-z))


def softmax(scores):
    scores = np.asarray(scores, dtype=float)
    shifted = scores - np.max(scores)
    exps = np.exp(shifted)
    probs = exps / np.sum(exps)
    return [round(float(p), 4) for p in probs]


def single_neuron_model(features, labels, weights, bias):
    features = np.asarray(features, dtype=float)
    labels = np.asarray(labels, dtype=float)
    weights = np.asarray(weights, dtype=float)
    z = features @ weights + bias
    probabilities = sigmoid(z)
    mse = float(np.mean((probabilities - labels) ** 2))
    return [round(float(p), 4) for p in probabilities], round(mse, 4)


def relu(z):
    z = np.asarray(z, dtype=float)
    return np.maximum(0.0, z)


def leaky_relu(z, alpha=0.01):
    z = np.asarray(z, dtype=float)
    return np.where(z > 0, z, alpha * z)


def hard_sigmoid(x):
    x = np.asarray(x, dtype=float)
    return np.clip(0.2 * x + 0.5, 0.0, 1.0)


def elu(x, alpha=1.0):
    x = np.asarray(x, dtype=float)
    with np.errstate(over='ignore'):
        return np.where(x > 0, x, alpha * (np.exp(np.minimum(x, 0)) - 1))


def prelu(x, alpha=0.25):
    x = np.asarray(x, dtype=float)
    return np.where(x >= 0, x, alpha * x)


def softplus(x):
    x = np.asarray(x, dtype=float)
    with np.errstate(over='ignore'):
        return np.maximum(x, 0) + np.log1p(np.exp(-np.abs(x)))


def softsign(x):
    x = np.asarray(x, dtype=float)
    return x / (1.0 + np.abs(x))


def swish(x):
    x = np.asarray(x, dtype=float)
    return x * sigmoid(x)


def selu(x):
    x = np.asarray(x, dtype=float)
    alpha = 1.6732632423543772
    scale = 1.0507009873554804
    with np.errstate(over='ignore'):
        return scale * np.where(x > 0, x, alpha * (np.exp(np.minimum(x, 0)) - 1))
```

</details>